# AirfRANS GNN Surrogate — Kaggle GPU setup

Fallback for when Colab's free GPU quota runs out -- separate quota pool
(~30 hrs/week of P100 or T4x2). No Drive-equivalent live mount here: Kaggle
persists across sessions via **Datasets** (read-only, attached to a session)
and a notebook's own **Output** (from "Save Version"), not a synced folder.

Before running: Settings (right sidebar) > Accelerator > GPU, and
Internet > On (needed for git clone / pip install / dataset download).

## 0. This run tests the loss formulation, not the architecture

The normals-feature run (previous version of this notebook) was stopped
partway through -- with `wall_weight_peak=20` (the distance-weighted loss,
`src/train.py`), since the paper's own baselines on this exact dataset
(GraphSAGE, Graph U-Net -- comparable-caliber GNN architectures) get Cd
relative error 4-15% with what's presumably a standard, unweighted loss,
while our best checkpoint so far gets ~203% by the same metric (see
`ARCHITECTURE.md` section 11 and the paper comparison in project history).
That gap is too large to blame on the architecture when similar
architectures do fine on the same data -- the most likely single culprit is
the custom distance-weighted loss: it rewards fitting the near-wall
*value*, not the *gradient* that friction drag actually depends on, and its
"near-wall" length scale covers 61% of a typical mesh, not a tight
boundary-layer band.

This run isolates that variable: same architecture, same normals feature,
**plain unweighted MSE** (`wall_weight_peak=1.0`, which reduces the
weighting formula `1 + (peak-1)*exp(...)` to a constant 1 everywhere) --
the more standard recipe the paper's baselines likely used. If Cd comes
back much closer to their 4-15% range, the distance-weighted loss was the
problem, not the architecture. If it's still wildly worse, that's real
evidence to reconsider architecture or training duration next.

**A real bug, found and fixed, in how checkpoint isolation used to work:**
`main()`'s periodic checkpoints (`mgn-{epoch:03d}.ckpt`) always save into
`os.path.dirname(checkpoint_path)` -- previous versions of this notebook
picked a new *filename* per run (e.g. `meshgraphnet_normals.ckpt`) but left
it directly under `/kaggle/working`, the same directory every run has used.
Changing the filename doesn't change that directory, so auto-resume (which
globs `mgn-epoch=*.ckpt` in that directory) can silently pick up a
*previous* run's checkpoints if they're still sitting there in the same
session -- the "distinct path" was only distinct in name, not in the
directory that actually matters. Fixed below by giving this run its own
**subdirectory**, not just a new filename.

**Continuing the same Kaggle session instead of restarting:** if your
cache (`CACHE_DIR` below) is already populated from the normals run, you
don't need to re-download/re-preprocess. Skip straight to running
`!git -C /kaggle/working/repo pull` (not a fresh `git clone` -- that fails
into a non-empty directory) to pick up this notebook's code changes, then
run the checkpoint-path cell and training cell below. Only re-run cells
5-7 if you restarted the session (cache wiped) or never ran them this
session.</cell id="cell-1">


In [ ]:
# A dedicated SUBDIRECTORY per run, not just a new filename in the shared
# /kaggle/working -- os.path.dirname(checkpoint_path) is what auto-resume
# actually globs for "mgn-epoch=*.ckpt", so two runs sharing a directory can
# silently cross-contaminate regardless of how differently their checkpoint
# *files* are named (see section 0 above -- this bit a real run).
CHECKPOINT_PATH = "/kaggle/working/runs/plainmse/meshgraphnet.ckpt"

import glob
import os

existing = glob.glob(os.path.join(os.path.dirname(CHECKPOINT_PATH), "mgn-epoch=*.ckpt"))
print("existing checkpoints in this run's OWN directory (should be empty for a fresh start):", existing)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo and install dependencies

Same as Colab -- Kaggle also ships CUDA-enabled torch preinstalled, and
`torch_geometric` installs as pure Python (no `torch-scatter`/`torch-sparse`
needed, confirmed working on both machines already).

In [ ]:
REPO_URL = "https://github.com/Revanthkr1/airfrans-gnn-surrogate.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q torch_geometric lightning airfrans pyvista

## 2. Download + preprocess, one case at a time

`af.dataset.download(unzip=True)` needs the zip (~9.34GB) *and* the fully
extracted dataset (~15GB) on disk simultaneously to extract everything at
once -- that alone exceeded this session's disk quota (`OSError: No space
left on device`, mid-extraction, before preprocessing even started).

Instead: download just the zip, then extract + cache one case at a time,
deleting each case's raw files immediately after caching. Peak disk usage
stays at roughly (zip + one case + the cache built so far) instead of
(zip + the entire raw dataset) at once. `manifest.json` doesn't need
extracting from the zip either -- it's committed to the repo at
`data/manifest.json`.

In [ ]:
from src.data import split_names
from src.preprocess import download_zip_only, stream_preprocess_from_zip

DATA_ROOT = "data"
WORK_DIR = "/kaggle/working/raw"  # transient extraction scratch, one case at a time
CACHE_DIR = "/kaggle/working/cache/full"
MANIFEST_DIR = "data"  # data/manifest.json is committed to the repo -- always present

train_names = split_names(MANIFEST_DIR, task="full", train=True)
zip_path = download_zip_only(DATA_ROOT)
stream_preprocess_from_zip(zip_path, train_names, CACHE_DIR, WORK_DIR)

import os
os.remove(zip_path)  # done with it -- frees ~9.34GB back
print(f"cached {len(os.listdir(CACHE_DIR))}/{len(train_names)} training cases")

## 3. Train, fresh, with plain (unweighted) MSE

`wall_weight_peak=1.0` is passed explicitly below -- this collapses
`distance_weighted_mse`'s weight formula to a constant 1 for every node,
i.e. plain MSE, same as `torch.nn.functional.mse_loss`. Everything else
matches the normals run: `node_in_dim=7` picked up automatically via
`model_kwargs=None`, `batch_size=1` + `accumulate_grad_batches=4`,
`precision="16-mixed"`, `checkpoint_every_n_epochs=5`.

Once this finishes (or periodically during it), evaluate against the same
paper-comparable metrics used in project history (`cd_mean_rel_err`,
`cl_mean_rel_err`, `cd_spearman`, `cl_spearman` in `src/evaluate.py`) and
compare directly to the weighted-loss run's epoch 47 (Cd mean rel. error
202.73%, Cl 49.34%) and to the AirfRANS paper's own baselines (Cd 4-15%,
Cl 0.5-0.8%) -- that comparison is the actual point of this run.

**To persist past this session**: click "Save Version" when done (or
periodically).</cell id="cell-8">


In [ ]:
from src.train import main as train_main

train_main(
    dataset_root=MANIFEST_DIR,
    cache_dir=CACHE_DIR,
    stats_path="data/norm_stats.npz",
    checkpoint_path=CHECKPOINT_PATH,
    max_epochs=100,
    batch_size=1,
    accumulate_grad_batches=4,
    n_val=80,
    checkpoint_every_n_epochs=5,
    num_workers=2,
    precision="16-mixed",
    wall_weight_peak=1.0,  # plain MSE -- see section 0 for why
)